In [ ]:
import json
import os
import re

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from matplotlib import pyplot as plt


In [ ]:
model = "Qwen/Qwen2.5-Coder-32B"
short_model_name = model.split("/")[-1]

model_results_dir = f"../model_results/{short_model_name}"
model_aoi_comparison_results_dir = os.path.join(model_results_dir, "data_driven_aois_comparison")
model_individual_results_dir = os.path.join(model_results_dir, "individual_analysis")
output_dir = "RQ3"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
analysis_df = pd.read_csv(os.path.join(model_individual_results_dir, "eeg_perplexity_analysis_individual.csv"))
analysis_df["snippet_id"] = analysis_df["file"].str.extract(r"(.*)-\d+$")[0]
# read overlapping results from aoi_comparison
file_path = os.path.join(model_aoi_comparison_results_dir, "aoi_overlap_snippet_lists.json")
with open(file_path, 'r') as f:
    aoi_comparison = json.load(f)

In [ ]:
def create_eeg_perplexity_correlation_plots_individual(analysis_df):
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    overlapping_clean = aoi_comparison["overlap"]["clean"]
    overlapping_obf = aoi_comparison["overlap"]["obf"]
    non_overlapping_clean = aoi_comparison["no_overlap"]["clean"]
    non_overlapping_obf = aoi_comparison["no_overlap"]["obf"]

    correlation_results = []

    # Automatically detect relevant columns
    perplexity_metrics = [col for col in analysis_df.columns if col.endswith('Max AOI Perplexity')]
    eeg_metrics = [col for col in analysis_df.columns if col.endswith('mean_eeg')]

    def make_numbered_mask(df, base_list):
        # Build regex for all base patterns, matching only files that start with the base and end with -1..-10 before extension
        regexes = [rf"^{re.escape(base)}-(?:[1-9]|10)" for base in base_list]
        combined_regex = "|".join(regexes)
        return df['file'].str.contains(combined_regex, regex=True)

    # Define subsets: full, clean, and obf
    subsets = {
        "full": analysis_df,
        "clean": analysis_df[analysis_df['file'].str.contains("clean", case=False, na=False)],
        "obf": analysis_df[analysis_df['file'].str.contains("obf", case=False, na=False)],
        "overlapping_clean": analysis_df[make_numbered_mask(analysis_df, overlapping_clean)],
        "overlapping_obf": analysis_df[make_numbered_mask(analysis_df, overlapping_obf)],
        "non_overlapping_clean": analysis_df[make_numbered_mask(analysis_df, non_overlapping_clean)],
        "non_overlapping_obf": analysis_df[make_numbered_mask(analysis_df, non_overlapping_obf)],
    }

    # Define AOI categories and their colors
    category_colors = {
        "Declaration": "#1f77b4",  # blue
        "Variable / Assignment": "#ff7f0e",  # orange
        "Type": "#2ca02c",  # green
        "Expression": "#d62728",  # red
        "Function Call": "#9467bd",  # purple
        "Control Flow": "#8c564b",  # brown
        "Exception Handling": "#e377c2",  # pink
        "Statement / Block": "#7f7f7f",  # gray
        "Identifier": "#bcbd22",  # olive
        "Literal": "#17becf",  # cyan
        "Operator": "#aec7e8",  # light blue
        "Modifier": "#ffbb78",  # light orange
        "Annotation": "#98df8a",  # light green
        "Import / Package": "#c5b0d5",  # light purple
        "Parameter / Argument": "#c49c94",  # light brown
        "Generic": "#f7b6d2",  # light pink
        "Comment": "#c7c7c7",  # light gray
        "Program Structure": "#dbdb8d",  # khaki
        "Punctuation": "#9edae5",  # light cyan
        "Other": "#000000",  # black
    }

    for subset_name, subset_df in subsets.items():
        for perp_metric in perplexity_metrics:
            safe_perp_name = perp_metric.replace(' ', '_').lower()
            for eeg_metric in eeg_metrics:
                eeg_prefix = eeg_metric.replace('_eeg', ' ').capitalize() + 'EEG Signal'
                plt.figure(figsize=(10, 8))
                x_vals = subset_df[perp_metric]
                y_vals = pd.to_numeric(subset_df[eeg_metric], errors='coerce')
                categories = subset_df['aoi_category']

                # Filter out NaN
                valid_indices = x_vals.notna() & y_vals.notna() & categories.notna()
                x_vals = x_vals[valid_indices].reset_index(drop=True)
                y_vals = y_vals[valid_indices].reset_index(drop=True)
                categories = categories[valid_indices].reset_index(drop=True)

                if x_vals.empty or y_vals.empty:
                    print(f"[{subset_name}] Skipping {eeg_prefix} vs {perp_metric} due to empty data.")
                    continue

                # Plot each category separately
                for cat, color in category_colors.items():
                    mask = categories == cat
                    if mask.any():
                        plt.scatter(
                            x_vals[mask], y_vals[mask],
                            color=color, alpha=0.7, s=100, label=cat
                        )

                # Log scale
                plt.xscale('log', base=10)
                valid_x = x_vals[x_vals > 0]
                if valid_x.empty:
                    continue
                min_exp = int(np.floor(np.log10(valid_x.min())))
                max_exp = int(np.ceil(np.log10(valid_x.max())))
                xticks = [10 ** i for i in range(min_exp, max_exp + 1)]
                xtick_labels = [r"$10^{%d}$" % i for i in range(min_exp, max_exp + 1)]
                plt.xticks(xticks, xtick_labels)

                # Trend line in log space
                x_log = np.log(valid_x)
                y_trimmed = y_vals.loc[valid_x.index]
                if len(x_log) > 1:
                    # Linear fit
                    z = np.polyfit(x_log, y_trimmed, 1)
                    p = np.poly1d(z)
                    x_range = np.linspace(min(x_log), max(x_log), 100)
                    plt.plot(np.exp(x_range), p(x_range), "r--", alpha=0.7, label='Trend Line')

                    # Correlations
                    spearman_r, spearman_p = spearmanr(x_vals, y_vals)
                else:
                    spearman_r = spearman_p = np.nan

                correlation_results.append({
                    'subset': subset_name,
                    'eeg_metric': eeg_metric,
                    'perplexity_metric': perp_metric,
                    'spearman_r': spearman_r,
                    'spearman_p': spearman_p
                })

                plt.title(f'{subset_name.upper()}: {eeg_prefix} vs {perp_metric} (Log Scale)\n'
                          f'Spearman r={spearman_r:.3f} (p={spearman_p:.3f})'
                          )
                plt.xlabel(f'{perp_metric} (Log Scale)')
                plt.ylabel(f'{eeg_prefix}')
                plt.grid(True, alpha=0.3)
                plt.legend(title="AOI Category", bbox_to_anchor=(1.05, 1), loc='upper left')

                filename = f"{eeg_metric}_vs_{safe_perp_name}_{subset_name}_log.pdf"
                plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')

                plt.close()

    correlation_df = pd.DataFrame(correlation_results)
    correlation_csv_path = os.path.join(output_dir, "correlation_statistics_individual_by_subset.csv")
    correlation_df.to_csv(correlation_csv_path, index=False)
    print(f"Correlation statistics saved to {correlation_csv_path}")

    return analysis_df, correlation_df

In [ ]:
create_eeg_perplexity_correlation_plots_individual(analysis_df=analysis_df)


In [ ]:
from scipy.stats import spearmanr
import numpy as np


def clustered_bootstrap_spearman(df, x_col, y_col, cluster_col, n_iter=10000, seed=42):
    np.random.seed(seed)
    cluster_ids = df[cluster_col].unique()
    rhos = []

    for _ in range(n_iter):
        sampled_ids = np.random.choice(cluster_ids, size=len(cluster_ids), replace=True)
        sampled_df = pd.concat([df[df[cluster_col] == cid] for cid in sampled_ids])

        x = pd.to_numeric(sampled_df[x_col], errors='coerce')
        y = pd.to_numeric(sampled_df[y_col], errors='coerce') * 1e6
        valid = (x > 0) & y.notna()
        if valid.sum() < 3:
            continue

        rho, _ = spearmanr(x[valid], y[valid])
        rhos.append(rho)

    rhos = np.array(rhos)
    rho_mean = np.mean(rhos)
    ci_low = np.percentile(rhos, 2.5)
    ci_high = np.percentile(rhos, 97.5)

    # Two-sided bootstrap p-value
    p_value = 2 * min(
        np.mean(rhos <= 0),
        np.mean(rhos >= 0)
    )

    return {
        "rho_mean": rho_mean,
        "95%_CI": (ci_low, ci_high),
        "p_value": p_value,
        "rhos": rhos
    }


def report_bootstrap_spearman(df, mask, group_name, x_col, y_col, cluster_col):
    x = pd.to_numeric(df.loc[mask, x_col], errors='coerce')
    y = pd.to_numeric(df.loc[mask, y_col], errors='coerce') * 1e6
    valid = (x > 0) & y.notna()
    df_filtered = df.loc[mask][valid]

    if len(df_filtered) < 3:
        print(f"{group_name}: Not enough valid data points for Spearman correlation.")
        return

    result = clustered_bootstrap_spearman(
        df_filtered, x_col=x_col, y_col=y_col, cluster_col=cluster_col
    )

    rho = result["rho_mean"]
    ci_low, ci_high = result["95%_CI"]
    p_value = result["p_value"]

    print(
        f"{group_name}: Spearman ρ = {rho:.2f}, "
        f"95% CI = [{ci_low:.2f}, {ci_high:.2f}], "
        f"p = {p_value:.3f}"
    )


In [ ]:
def plot_clean_confusing_with_overlap_distinction(analysis_df):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import re

    plt.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})

    def make_numbered_mask(df, base_list):
        regexes = [rf"^{re.escape(base)}-(?:[1-9]|10)" for base in base_list]
        combined_regex = "|".join(regexes)
        return df['file'].str.contains(combined_regex, regex=True)

    clean_df = analysis_df

    overlapping_clean = make_numbered_mask(clean_df, aoi_comparison["overlap"]["clean"])
    non_overlapping_clean = make_numbered_mask(clean_df, aoi_comparison["no_overlap"]["clean"])
    overlapping_obf = make_numbered_mask(clean_df, aoi_comparison["overlap"]["obf"])
    non_overlapping_obf = make_numbered_mask(clean_df, aoi_comparison["no_overlap"]["obf"])

    x_col = 'Max AOI Perplexity'
    y_col = 'mean_eeg'

    def extract_xy(df, mask):
        x = pd.to_numeric(df.loc[mask, x_col], errors='coerce')
        y = pd.to_numeric(df.loc[mask, y_col], errors='coerce') * 1e6
        valid = (x > 0) & y.notna()
        return x[valid], y[valid]

    clean_overlap_x, clean_overlap_y = extract_xy(clean_df, overlapping_clean)
    clean_nooverlap_x, clean_nooverlap_y = extract_xy(clean_df, non_overlapping_clean)
    obf_overlap_x, obf_overlap_y = extract_xy(clean_df, overlapping_obf)
    obf_nooverlap_x, obf_nooverlap_y = extract_xy(clean_df, non_overlapping_obf)

    # Shared x-axis ticks
    x_vals = pd.concat([clean_overlap_x, clean_nooverlap_x, obf_overlap_x, obf_nooverlap_x])
    x_min = x_vals.min()
    x_max = x_vals.max()
    min_exp = 0
    max_exp = int(np.ceil(np.log10(x_max)))
    xticks = [10 ** i for i in range(min_exp, max_exp + 1)]

    fig, axs = plt.subplots(1, 2, figsize=(6, 2), sharey=True)

    def plot_panel(ax, title, df, mask1, mask2, snippet_type, x_col, y_col, cluster_col):
        color = "#1F77B4" if snippet_type == "clean" else "#D62728"

        def extract_xy_mask(df, mask):
            x = pd.to_numeric(df.loc[mask, x_col], errors='coerce')
            y = pd.to_numeric(df.loc[mask, y_col], errors='coerce') * 1e6
            valid = (x > 0) & y.notna()
            return df.loc[mask][valid], x[valid], y[valid]

        df1, x1, y1 = extract_xy_mask(df, mask1)
        df2, x2, y2 = extract_xy_mask(df, mask2)

        ax.scatter(x1, y1, s=25, alpha=0.6, color=color, marker='o', label='Overlap')
        ax.scatter(x2, y2, s=25, alpha=0.6, color=color, marker='x', label='No Overlap')

        # Trend line over combined data
        x_all = pd.concat([x1, x2])
        y_all = pd.concat([y1, y2])
        x_log = np.log(x_all)
        coeffs = np.polyfit(x_log, y_all, 1)
        poly_fn = np.poly1d(coeffs)
        x_range = np.linspace(np.log(x_all.min()), np.log(x_all.max()), 100)
        ax.plot(np.exp(x_range), poly_fn(x_range), linestyle='-', color=color, linewidth=1)

        # Bootstrapped Spearman for all AOIs in this group
        df_combined = pd.concat([df1, df2])
        boot_result = clustered_bootstrap_spearman(
            df_combined, x_col=x_col, y_col=y_col, cluster_col=cluster_col
        )
        rho_mean = boot_result["rho_mean"]
        ci_low, ci_high = boot_result["95%_CI"]
        text = rf"Spearman $\rho$ = {rho_mean:.2f}" + "\n" + rf"[95% CI: {ci_low:.2f}, {ci_high:.2f}]"

        ax.text(1.05, 0.04, text, transform=ax.transAxes,
                verticalalignment='bottom', horizontalalignment='right',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.4, pad=2))

        ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xscale('log', base=10)
        ax.set_xlabel("Max AOI Perplexity")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    report_bootstrap_spearman(clean_df, overlapping_clean, "Overlap Clean", x_col, y_col, "snippet_id")
    report_bootstrap_spearman(clean_df, non_overlapping_clean, "No-Overlap Clean", x_col, y_col, "snippet_id")
    report_bootstrap_spearman(clean_df, overlapping_obf, "Overlap Confusing", x_col, y_col, "snippet_id")
    report_bootstrap_spearman(clean_df, non_overlapping_obf, "No-Overlap Confusing", x_col, y_col, "snippet_id")

    plot_panel(axs[0], "Confusing Snippets", analysis_df, overlapping_obf, non_overlapping_obf,
               snippet_type="confusing", x_col=x_col, y_col=y_col, cluster_col="snippet_id")
    plot_panel(axs[1], "Clean Snippets", analysis_df, overlapping_clean, non_overlapping_clean, snippet_type="clean",
               x_col=x_col, y_col=y_col, cluster_col="snippet_id")

    axs[0].set_ylabel(r"Mean FRP ($\mu$V)")
    xticks = [10 ** i for i in range(8)]
    xticklabels = [f"$10^{i}$" for i in range(8)]

    for ax in axs:
        ax.set_xscale('log', base=10)
        ax.set_xlim(10 ** 0, 10 ** 7)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels)
        ax.set_ylim(-10, 12)
        ax.set_yticks(np.arange(-10, 11, 10))

    # Custom legend below both subplots
    scatter_handles = [
        plt.Line2D([0], [0], marker='o', color='gray', linestyle='', markersize=6, label='Overlap with AoC'),
        plt.Line2D([0], [0], marker='x', color='gray', linestyle='', markersize=6, label='No Overlap with AoC')
    ]

    all_handles = scatter_handles
    all_labels = [h.get_label() for h in all_handles]

    fig.legend(
        handles=all_handles,
        labels=all_labels,
        loc='lower center',
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, -0.1)
    )

    plt.tight_layout()
    filename = f"{output_dir}/clean_vs_confusing_overlap_vs_nooverlap.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300, pad_inches=0)
    plt.show()


In [ ]:
plot_clean_confusing_with_overlap_distinction(analysis_df=analysis_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_indicators

plt.rcParams.update({'font.size': 9, 'font.family': 'sans-serif'})

# --- Build data ---
data_rows = []
statistics_path = os.path.join(model_aoi_comparison_results_dir, "data_driven_aois_overlap_statistics.json")
statistics = json.load(open(statistics_path, "r"))

# Original groups (without Detected Regions)
for i in range(statistics["num_confusing_only"]):
    data_rows.append({
        'region_id': f'confusing_only_{i}',
        'Manually Labeled Clean AoC': False,
        'Manually Labeled Conf. AoC': True,
        'Detected Region in Clean Snippet': False,
        'Detected Region in Conf. Snippet': False
    })

for i in range(statistics["num_clean_only"]):
    data_rows.append({
        'region_id': f'clean_only_{i}',
        'Manually Labeled Clean AoC': True,
        'Manually Labeled Conf. AoC': False,
        'Detected Region in Clean Snippet': False,
        'Detected Region in Conf. Snippet': False
    })

# Overlapping groups (with Detected Regions)
for i in range(statistics["num_clean_overlap"]):
    data_rows.append({
        'region_id': f'clean_overlap_{i}',
        'Manually Labeled Clean AoC': True,
        'Manually Labeled Conf. AoC': False,
        'Detected Region in Clean Snippet': True,
        'Detected Region in Conf. Snippet': False
    })

for i in range(statistics["num_confusing_overlap"]):
    data_rows.append({
        'region_id': f'confusing_overlap_{i}',
        'Manually Labeled Clean AoC': False,
        'Manually Labeled Conf. AoC': True,
        'Detected Region in Clean Snippet': False,
        'Detected Region in Conf. Snippet': True
    })

# Full-snippet entries (not overlapping with AoC)
for i in range(statistics["num_total_clean_detected"] - statistics["num_clean_overlap"]):
    data_rows.append({
        'region_id': f'in_clean_snippet_only_{i}',
        'Manually Labeled Clean AoC': False,
        'Manually Labeled Conf. AoC': False,
        'Detected Region in Clean Snippet': True,
        'Detected Region in Conf. Snippet': False
    })
for i in range(statistics["num_total_confusing_detected"] - statistics["num_confusing_overlap"]):
    data_rows.append({
        'region_id': f'in_confusing_snippet_only_{i}',
        'Manually Labeled Clean AoC': False,
        'Manually Labeled Conf. AoC': False,
        'Detected Region in Clean Snippet': False,
        'Detected Region in Conf. Snippet': True
    })

# Create DataFrame
df = pd.DataFrame(data_rows)

# Ensure all columns are boolean
for col in ['Manually Labeled Clean AoC', 'Manually Labeled Conf. AoC', 'Detected Region in Clean Snippet',
            'Detected Region in Conf. Snippet']:
    df[col] = df[col].astype(bool)

# Create UpSet data (no Detected Regions)
columns = ['Manually Labeled Clean AoC', 'Manually Labeled Conf. AoC', 'Detected Region in Clean Snippet',
           'Detected Region in Conf. Snippet']
upset_data = from_indicators(df[columns])

# Create MultiIndex Series for intersections
intersection_df = upset_data.to_frame().reset_index()
# Add a custom order column for sorting
custom_sort_order = {
    (False, True, False, False): 2,
    (True, False, False, False): 3,
    (False, False, True, False): 1,
    (False, False, False, True): 0,
    (True, False, True, False): 4,
    (False, True, False, True): 5,
}

intersection_df['custom_order'] = intersection_df.apply(
    lambda row: custom_sort_order.get(tuple(row[col] for col in columns), 999), axis=1
)

# Sort the DataFrame
intersection_df = intersection_df.sort_values(by='custom_order')

# Convert back to MultiIndex Series
intersection_df = intersection_df.set_index(columns)
upset_data_sorted = intersection_df

# Create the plot
fig = plt.figure(figsize=(3.2, 1.7))
upset = UpSet(
    upset_data_sorted,
    subset_size='count',
    show_counts=True,
    sort_by='input',
    sort_categories_by='-input',
    element_size=25,
    intersection_plot_elements=3,
    facecolor='gray',
    with_lines=True
)
axes = upset.plot(fig=fig)

# --- Define color codes ---
clean_color = "#1F77B4CC"  # Dark Blue
confusing_color = "#D62728CC"  # Red
overlap_color = "#9467BDCC"  # Purple

# --- Color total bars based on label text ---
# Map labels to bars
category_order = ["gray", "gray", confusing_color, clean_color]
for bar, color in zip(axes['totals'].patches, category_order):
    bar.set_facecolor(color)

# --- Color intersection bars based on label text ---
color_order = ["gray", "gray", confusing_color, clean_color, overlap_color, overlap_color]
for bar, label in zip(axes['intersections'].patches, color_order):
    bar.set_facecolor(label)

# --- Clean up outlines ---
for bar in axes['intersections'].patches:
    bar.set_edgecolor("none")
for bar in axes['totals'].patches:
    bar.set_edgecolor("none")

# --- Clean axis appearance ---
intersection_ax = axes['intersections']
intersection_ax.set_ylabel("Intersection\nsize")
intersection_ax.grid(False)
for spine in intersection_ax.spines.values():
    if spine.spine_type != 'bottom':
        spine.set_visible(False)
intersection_ax.set_yticks([])

totals_ax = axes['totals']
totals_ax.grid(False)
for spine in totals_ax.spines.values():
    if spine.spine_type != 'left':
        spine.set_visible(False)
totals_ax.set_xticks([])

# --- Save and show ---
filename = f"{output_dir}/upset_plot_overlap.pdf"
plt.savefig(filename, bbox_inches='tight', dpi=300, pad_inches=0)
plt.show()
